[Reference](https://levelup.gitconnected.com/building-a-gpt-4o-like-multi-modal-from-scratch-using-python-ad0fa9c213d3)

In [1]:
# Core PyTorch libraries for building models, tensors, and training
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Image processing and computer vision
import torchvision
import torchvision.transforms as transforms
from PIL import Image, ImageDraw # For creating/handling dummy images

# Standard Python libraries
import re               # For regular expressions (used in BPE word splitting)
import collections      # For data structures like Counter (used in BPE frequency counting)
import math             # For mathematical functions (used in Positional Encoding)
import os               # For interacting with the operating system (file paths, directories)

# Numerical Python (Used conceptually for audio waveform generation)
import numpy as np

# Coding a BPE Tokenizer

In [2]:
# Define the raw text corpus for training the BPE tokenizer
corpus_raw = """
Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it had no pictures or conversations in
it, 'and what is the use of a book,' thought Alice 'without pictures or
conversation?'
So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy and stupid), whether the pleasure
of making a daisy-chain would be worth the trouble of getting up and
picking the daisies, when suddenly a White Rabbit with pink eyes ran
close by her.
"""

In [3]:
# Convert the raw corpus to lowercase
corpus_lower = corpus_raw.lower()

In [4]:
# Define the regular expression for splitting words and punctuation
split_pattern = r'\w+|[^\s\w]+'

# Apply the regex to the lowercased corpus to get a list of initial tokens
initial_word_list = re.findall(split_pattern, corpus_lower)

print(f"Corpus split into {len(initial_word_list)} initial words/tokens.")

# Display the first few tokens
print(f"First 3 initial tokens: {initial_word_list[:3]}")

Corpus split into 127 initial words/tokens.
First 3 initial tokens: ['alice', 'was', 'beginning']


In [5]:
# Use collections.Counter to count frequencies of items in initial_word_list
word_frequencies = collections.Counter(initial_word_list)

# Display the 3 most frequent tokens and their counts
print("3 Most frequent tokens:")
for token, count in word_frequencies.most_common(3):
    print(f"  '{token}': {count}")

3 Most frequent tokens:
  'the': 7
  'of': 5
  'her': 5


In [6]:
# Define the special end-of-word symbol "</w>"
end_of_word_symbol = '</w>'

# Create a dictionary to hold the initial representation of the corpus
# Key: original unique word/token, Value: list of characters + end_of_word_symbol
initial_corpus_representation = {}

# Iterate through the unique words/tokens identified by the frequency counter
for word in word_frequencies:
    # Convert the word string into a list of its characters
    char_list = list(word)
    # Append the end-of-word symbol to the list
    char_list.append(end_of_word_symbol)
    # Store this list in the dictionary with the original word as the key
    initial_corpus_representation[word] = char_list

In [7]:
# Display the representation for a sample word
example_word = 'beginning'
if example_word in initial_corpus_representation:
    print(f"Representation for '{example_word}': {initial_corpus_representation[example_word]}")
example_punct = '.'
if example_punct in initial_corpus_representation:
    print(f"Representation for '{example_punct}': {initial_corpus_representation[example_punct]}")

Representation for 'beginning': ['b', 'e', 'g', 'i', 'n', 'n', 'i', 'n', 'g', '</w>']
Representation for '.': ['.', '</w>']


In [8]:
# Initialize an empty set to store the unique initial symbols (vocabulary)
initial_vocabulary = set()

# Iterate through the character lists stored in the initial corpus representation
for word in initial_corpus_representation:
    # Get the list of symbols for the current word
    symbols_list = initial_corpus_representation[word]
    # Update the vocabulary set with the symbols from this list
    # The `update` method adds all elements from an iterable (like a list) to the set
    initial_vocabulary.update(symbols_list)

# Although update should have added '</w>', we can explicitly add it for certainty
# initial_vocabulary.add(end_of_word_symbol)

print(f"Initial vocabulary created with {len(initial_vocabulary)} unique symbols.")
# Optional: Display the sorted list of initial vocabulary symbols
print(f"Initial vocabulary symbols: {sorted(list(initial_vocabulary))}")

Initial vocabulary created with 31 unique symbols.
Initial vocabulary symbols: ["'", '(', ')', ',', '-', '.', ':', '</w>', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']


In [9]:
# Define the desired number of merge operations
# This determines how many new subword tokens will be added to the initial character vocab
num_merges = 75 # Let's use 75 merges for this example

# Initialize an empty dictionary to store the learned merge rules
# Format: { (symbol1, symbol2): merge_priority_index }
learned_merges = {}

# Create a working copy of the corpus representation to modify during training
# Using .copy() ensures we don't alter the original initial_corpus_representation
current_corpus_split = initial_corpus_representation.copy()

# Create a working copy of the vocabulary to modify during training
current_vocab = initial_vocabulary.copy()

print(f"Training state initialized. Target number of merges: {num_merges}")
print(f"Initial vocabulary size: {len(current_vocab)}")


Training state initialized. Target number of merges: 75
Initial vocabulary size: 31


In [10]:
# Start the main loop that iterates for the specified number of merges
print(f"\n--- Starting BPE Training Loop ({num_merges} iterations) ---")
for i in range(num_merges):
    print(f"\nIteration {i + 1}/{num_merges}")

    # Calculate pair statistics
    pair_counts = collections.Counter()
    for word, freq in word_frequencies.items():
        symbols = current_corpus_split[word]
        for j in range(len(symbols) - 1):
            pair = (symbols[j], symbols[j+1])
            pair_counts[pair] += freq
    if not pair_counts:
        print("No more pairs found to merge. Stopping early.")
        break

    # Find the best pair
    best_pair = max(pair_counts, key=pair_counts.get)
    print(f"Found best pair: {best_pair} with frequency {pair_counts[best_pair]}")

    # Store merge rule
    learned_merges[best_pair] = i

    # Create new symbol
    new_symbol = "".join(best_pair)

    # Update corpus representation
    next_corpus_split = {}
    for word in current_corpus_split:
        old_symbols = current_corpus_split[word]
        new_symbols = []
        k = 0
        while k < len(old_symbols):
            if k < len(old_symbols) - 1 and (old_symbols[k], old_symbols[k+1]) == best_pair:
                new_symbols.append(new_symbol)
                k += 2
            else:
                new_symbols.append(old_symbols[k])
                k += 1
        next_corpus_split[word] = new_symbols
    current_corpus_split = next_corpus_split

    # Update vocabulary
    current_vocab.add(new_symbol)

# Final output
print(f"\n--- BPE Training Loop Finished after {i + 1} iterations ---")
final_vocabulary = current_vocab
final_learned_merges = learned_merges
final_corpus_representation = current_corpus_split


--- Starting BPE Training Loop (75 iterations) ---

Iteration 1/75
Found best pair: ('e', '</w>') with frequency 21

Iteration 2/75
Found best pair: ('i', 'n') with frequency 16

Iteration 3/75
Found best pair: ('e', 'r') with frequency 13

Iteration 4/75
Found best pair: ('t', 'h') with frequency 13

Iteration 5/75
Found best pair: ('d', '</w>') with frequency 12

Iteration 6/75
Found best pair: ('s', '</w>') with frequency 11

Iteration 7/75
Found best pair: ('in', 'g') with frequency 9

Iteration 8/75
Found best pair: ('ing', '</w>') with frequency 9

Iteration 9/75
Found best pair: ('t', '</w>') with frequency 9

Iteration 10/75
Found best pair: ('y', '</w>') with frequency 8

Iteration 11/75
Found best pair: ('er', '</w>') with frequency 8

Iteration 12/75
Found best pair: ('o', 'n') with frequency 7

Iteration 13/75
Found best pair: ('th', 'e</w>') with frequency 7

Iteration 14/75
Found best pair: ('i', 'c') with frequency 6

Iteration 15/75
Found best pair: ('o', '</w>') with 

In [11]:
print(f"Final Vocabulary Size: {len(final_vocabulary)} tokens")

Final Vocabulary Size: 106 tokens


In [12]:
# List some words we expect to see interesting tokenization for
example_words_to_inspect = ['beginning', 'conversations', 'sister', 'pictures', 'reading', 'alice']

for word in example_words_to_inspect:
    if word in final_corpus_representation:
        print(f"  '{word}': {final_corpus_representation[word]}")
    else:
        print(f"  '{word}': Not found in original corpus (should not happen if chosen from corpus).")

  'beginning': ['b', 'e', 'g', 'in', 'n', 'ing</w>']
  'conversations': ['conversati', 'on', 's</w>']
  'sister': ['sister</w>']
  'pictures': ['pictures</w>']
  'reading': ['re', 'ad', 'ing</w>']
  'alice': ['alice</w>']


In [13]:
# Define a new text string to tokenize using the learned BPE rules
# This text contains words seen in training ('alice', 'pictures')
# and potentially unseen words or variations ('tiresome', 'thought')
new_text_to_tokenize = "Alice thought reading was tiresome without pictures."

In [15]:
print(f"Original Input Text: '{new_text_to_tokenize}'")

Original Input Text: 'Alice thought reading was tiresome without pictures.'


# A Decoder-Only Transformer

In [16]:
# Define the raw text corpus for training
corpus_raw = """
Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it had no pictures or conversations in
it, 'and what is the use of a book,' thought Alice 'without pictures or
conversation?'
So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy and stupid), whether the pleasure
of making a daisy-chain would be worth the trouble of getting up and
picking the daisies, when suddenly a White Rabbit with pink eyes ran
close by her.
"""

In [17]:
# Find all unique characters in the raw corpus
chars = sorted(list(set(corpus_raw)))
vocab_size = len(chars)

# Create character-to-integer mapping (encoding)
char_to_int = { ch:i for i,ch in enumerate(chars) }

# Create integer-to-character mapping (decoding)
int_to_char = { i:ch for i,ch in enumerate(chars) }

print(f"Created character vocabulary of size: {vocab_size}")
print(f"Vocabulary: {''.join(chars)}")

Created character vocabulary of size: 36
Vocabulary: 
 '(),-.:?ARSWabcdefghiklmnoprstuvwy


In [18]:
# Encode the entire corpus into a list of integer IDs
encoded_corpus = [char_to_int[ch] for ch in corpus_raw]

# Convert the list into a PyTorch tensor
full_data_sequence = torch.tensor(encoded_corpus, dtype=torch.long)

print(f"Encoded corpus into a tensor of shape: {full_data_sequence.shape}")

Encoded corpus into a tensor of shape: torch.Size([593])


In [19]:
# Encode the entire corpus into a list of integer IDs
encoded_corpus = [char_to_int[ch] for ch in corpus_raw]

# Convert the list into a PyTorch tensor
full_data_sequence = torch.tensor(encoded_corpus, dtype=torch.long)

print(f"Encoded corpus into a tensor of shape: {full_data_sequence.shape}")

Encoded corpus into a tensor of shape: torch.Size([593])


In [20]:
# Define Model Hyperparameters (using calculated vocab_size)
# vocab_size = vocab_size # Already defined from data
d_model = 64         # Embedding dimension (vector size for each token)
n_heads = 4          # Number of attention heads (parallel attention calculations)
n_layers = 3         # Number of Transformer blocks stacked on top of each other
d_ff = d_model * 4   # Hidden dimension in the feed-forward networks
block_size = 32      # Maximum sequence length the model sees at once
# dropout_rate = 0.1 # Omitting dropout for simplicity

# Define Training Hyperparameters
learning_rate = 3e-4
batch_size = 16      # Number of sequences processed in parallel during training
epochs = 5000        # Number of training iterations
eval_interval = 500 # How often to print loss

# Device Configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Ensure d_model is divisible by n_heads
assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
d_k = d_model // n_heads # Dimension of keys/queries/values per head

print(f"Hyperparameters defined:")
print(f"  vocab_size: {vocab_size}")
print(f"  d_model: {d_model}")
print(f"  n_heads: {n_heads}")
print(f"  d_k (dim per head): {d_k}")
print(f"  n_layers: {n_layers}")
print(f"  d_ff: {d_ff}")
print(f"  block_size: {block_size}")
print(f"  learning_rate: {learning_rate}")
print(f"  batch_size: {batch_size}")
print(f"  epochs: {epochs}")
print(f"  Using device: {device}")

Hyperparameters defined:
  vocab_size: 36
  d_model: 64
  n_heads: 4
  d_k (dim per head): 16
  n_layers: 3
  d_ff: 256
  block_size: 32
  learning_rate: 0.0003
  batch_size: 16
  epochs: 5000
  Using device: cpu


In [21]:
# Create lists to hold all possible input (x) and target (y) sequences
all_x = []
all_y = []
num_total_tokens = len(full_data_sequence)
for i in range(num_total_tokens - block_size):
    x_chunk = full_data_sequence[i : i + block_size]
    y_chunk = full_data_sequence[i + 1 : i + block_size + 1]
    all_x.append(x_chunk)
    all_y.append(y_chunk)

# Stack the lists into tensors
train_x = torch.stack(all_x)
train_y = torch.stack(all_y)

num_sequences_available = train_x.shape[0]
print(f"Created {num_sequences_available} overlapping input/target sequence pairs.")
print(f"Shape of train_x: {train_x.shape}")
print(f"Shape of train_y: {train_y.shape}")

Created 561 overlapping input/target sequence pairs.
Shape of train_x: torch.Size([561, 32])
Shape of train_y: torch.Size([561, 32])


In [22]:
# Initialize the token embedding table
token_embedding_table = nn.Embedding(vocab_size, d_model).to(device)

print(f"Initialized Token Embedding Layer (Vocab: {vocab_size}, Dim: {d_model}). Device: {device}")

Initialized Token Embedding Layer (Vocab: 36, Dim: 64). Device: cpu


In [23]:
# Precompute the Sinusoidal Positional Encoding matrix
print("Creating Positional Encoding matrix...")
positional_encoding = torch.zeros(block_size, d_model, device=device)
position = torch.arange(0, block_size, dtype=torch.float, device=device).unsqueeze(1)
div_term_indices = torch.arange(0, d_model, 2, dtype=torch.float, device=device)
div_term = torch.exp(div_term_indices * (-math.log(10000.0) / d_model))
positional_encoding[:, 0::2] = torch.sin(position * div_term)
positional_encoding[:, 1::2] = torch.cos(position * div_term)
positional_encoding = positional_encoding.unsqueeze(0) # Add batch dimension

print(f"  Positional Encoding matrix created with shape: {positional_encoding.shape}. Device: {device}")

Creating Positional Encoding matrix...
  Positional Encoding matrix created with shape: torch.Size([1, 32, 64]). Device: cpu


In [24]:
print(f"Initializing components for {n_layers} Transformer layers...")

# Lists to store layers for each Transformer block
layer_norms_1 = []      # LayerNorm before MHA
layer_norms_2 = []      # LayerNorm before FFN
mha_qkv_linears = []    # Combined Linear layer for Q, K, V projections
mha_output_linears = [] # Output Linear layer for MHA
ffn_linear_1 = []       # First linear layer in FFN
ffn_linear_2 = []       # Second linear layer in FFN

for i in range(n_layers):
    # Layer Normalization 1 (for MHA input)
    layer_norms_1.append(nn.LayerNorm(d_model).to(device))
    # Multi-Head Attention: QKV projection
    mha_qkv_linears.append(nn.Linear(d_model, 3 * d_model, bias=False).to(device))
    # Multi-Head Attention: Output projection
    mha_output_linears.append(nn.Linear(d_model, d_model).to(device))
    # Layer Normalization 2 (for FFN input)
    layer_norms_2.append(nn.LayerNorm(d_model).to(device))
    # Feed-Forward Network: Layer 1
    ffn_linear_1.append(nn.Linear(d_model, d_ff).to(device))
    # Feed-Forward Network: Layer 2
    ffn_linear_2.append(nn.Linear(d_ff, d_model).to(device))
    print(f"  Initialized components for Layer {i+1}/{n_layers}.")

print(f"Finished initializing components for {n_layers} layers.")

Initializing components for 3 Transformer layers...
  Initialized components for Layer 1/3.
  Initialized components for Layer 2/3.
  Initialized components for Layer 3/3.
Finished initializing components for 3 layers.


In [25]:
print("Initializing final LayerNorm and Output layers...")

# Final Layer Normalization
final_layer_norm = nn.LayerNorm(d_model).to(device)
print(f"  Initialized Final LayerNorm. Device: {device}")

# Final Linear Layer (language modeling head)
output_linear_layer = nn.Linear(d_model, vocab_size).to(device)
print(f"  Initialized Output Linear Layer (to vocab size {vocab_size}). Device: {device}")

Initializing final LayerNorm and Output layers...
  Initialized Final LayerNorm. Device: cpu
  Initialized Output Linear Layer (to vocab size 36). Device: cpu


In [26]:
# Define the loss function
criterion = nn.CrossEntropyLoss()
print(f"Loss function defined: {type(criterion).__name__}")

# Gather all model parameters
all_model_parameters = list(token_embedding_table.parameters())
for i in range(n_layers):
    all_model_parameters.extend(list(layer_norms_1[i].parameters()))
    all_model_parameters.extend(list(mha_qkv_linears[i].parameters()))
    all_model_parameters.extend(list(mha_output_linears[i].parameters()))
    all_model_parameters.extend(list(layer_norms_2[i].parameters()))
    all_model_parameters.extend(list(ffn_linear_1[i].parameters()))
    all_model_parameters.extend(list(ffn_linear_2[i].parameters()))
all_model_parameters.extend(list(final_layer_norm.parameters()))
all_model_parameters.extend(list(output_linear_layer.parameters()))

# Define the AdamW optimizer
optimizer = optim.AdamW(all_model_parameters, lr=learning_rate)
print(f"Optimizer defined: {type(optimizer).__name__}")
print(f"  Managing {len(all_model_parameters)} parameter groups/tensors.")

# Create the causal mask ONCE (lower triangular matrix)
causal_mask = torch.tril(torch.ones(block_size, block_size, device=device)).view(1, 1, block_size, block_size)

Loss function defined: CrossEntropyLoss
Optimizer defined: AdamW
  Managing 38 parameter groups/tensors.


In [27]:
print(f"\nStarting Training Loop for {epochs} epochs...")
losses = []

# (Set layers to train mode - omitted for brevity, done in notebook)

for epoch in range(epochs):

    # --- 1. Batch Selection ---
    indices = torch.randint(0, num_sequences_available, (batch_size,))
    xb = train_x[indices].to(device) # (B, T)
    yb = train_y[indices].to(device) # (B, T)

    # --- 2. Forward Pass (Executing Steps 3.1-3.3 conceptually) ---
    # Simplified representation of the forward pass logic from the notebook:
    B, T = xb.shape
    C = d_model
    # Embed + Positional Encode
    token_embed = token_embedding_table(xb)
    pos_enc_slice = positional_encoding[:, :T, :]
    x = token_embed + pos_enc_slice
    # Transformer Blocks Loop
    for i in range(n_layers):
        x_input_block = x
        # Pre-LN MHA
        x_ln1 = layer_norms_1[i](x_input_block)
        qkv = mha_qkv_linears[i](x_ln1)
        qkv = qkv.view(B, T, n_heads, 3 * d_k).permute(0, 2, 1, 3)
        q, k, v = qkv.chunk(3, dim=-1)
        attn_scores = (q @ k.transpose(-2, -1)) * (d_k ** -0.5)
        attn_scores_masked = attn_scores.masked_fill(causal_mask[:,:,:T,:T] == 0, float('-inf'))
        attention_weights = F.softmax(attn_scores_masked, dim=-1)
        attn_output = attention_weights @ v
        attn_output = attn_output.permute(0, 2, 1, 3).contiguous().view(B, T, C)
        mha_result = mha_output_linears[i](attn_output)
        x = x_input_block + mha_result # Residual 1
        # Pre-LN FFN
        x_input_ffn = x
        x_ln2 = layer_norms_2[i](x_input_ffn)
        ffn_hidden = ffn_linear_1[i](x_ln2)
        ffn_activated = F.relu(ffn_hidden)
        ffn_output = ffn_linear_2[i](ffn_activated)
        x = x_input_ffn + ffn_output # Residual 2
    # Final Layers
    final_norm_output = final_layer_norm(x)
    logits = output_linear_layer(final_norm_output) # (B, T, vocab_size)

    # --- 3. Calculate Loss ---
    B_loss, T_loss, V_loss = logits.shape
    loss = criterion(logits.view(B_loss * T_loss, V_loss), yb.view(B_loss * T_loss))

    # --- 4. Zero Gradients ---
    optimizer.zero_grad()

    # --- 5. Backward Pass ---
    loss.backward()

    # --- 6. Update Parameters ---
    optimizer.step()

    # --- Logging ---
    current_loss = loss.item()
    losses.append(current_loss)
    if epoch % eval_interval == 0 or epoch == epochs - 1:
        print(f"  Epoch {epoch+1}/{epochs}, Loss: {current_loss:.4f}")

print("--- Training Loop Completed ---")


Starting Training Loop for 5000 epochs...
  Epoch 1/5000, Loss: 3.7172
  Epoch 501/5000, Loss: 0.4189
  Epoch 1001/5000, Loss: 0.1876
  Epoch 1501/5000, Loss: 0.1374
  Epoch 2001/5000, Loss: 0.1301
  Epoch 2501/5000, Loss: 0.1339
  Epoch 3001/5000, Loss: 0.1332
  Epoch 3501/5000, Loss: 0.1249
  Epoch 4001/5000, Loss: 0.1077
  Epoch 4501/5000, Loss: 0.1138
  Epoch 5000/5000, Loss: 0.1311
--- Training Loop Completed ---


# Generating Text Using Transformer

In [28]:
print("\n--- Step 5: Text Generation ---")

# Seed character(s)
seed_chars = "t"
seed_ids = [char_to_int[ch] for ch in seed_chars]
generated_sequence = torch.tensor([seed_ids], dtype=torch.long, device=device)
print(f"Initial seed sequence: '{seed_chars}' -> {generated_sequence.tolist()}")

# Define how many new tokens (characters) to generate
num_tokens_to_generate = 200
print(f"Generating {num_tokens_to_generate} new tokens...")

# (Set layers to eval mode - omitted for brevity, done in notebook)

# Disable gradient calculations for efficiency
with torch.no_grad():
    for _ in range(num_tokens_to_generate):
        # --- 1. Prepare Input Context (last block_size tokens) ---
        current_context = generated_sequence[:, -block_size:]

        # --- 2. Forward Pass (similar to training, using current context) ---
        # Simplified representation of the generation forward pass:
        B_gen, T_gen = current_context.shape
        C_gen = d_model
        token_embed_gen = token_embedding_table(current_context)
        pos_enc_slice_gen = positional_encoding[:, :T_gen, :]
        x_gen = token_embed_gen + pos_enc_slice_gen
        for i in range(n_layers):
             # Pre-LN MHA... (same logic as training)
             x_input_block_gen = x_gen
             x_ln1_gen = layer_norms_1[i](x_input_block_gen)
             qkv_gen = mha_qkv_linears[i](x_ln1_gen)
             qkv_gen = qkv_gen.view(B_gen, T_gen, n_heads, 3 * d_k).permute(0, 2, 1, 3)
             q_gen, k_gen, v_gen = qkv_gen.chunk(3, dim=-1)
             attn_scores_gen = (q_gen @ k_gen.transpose(-2, -1)) * (d_k ** -0.5)
             attn_scores_masked_gen = attn_scores_gen.masked_fill(causal_mask[:,:,:T_gen,:T_gen] == 0, float('-inf'))
             attention_weights_gen = F.softmax(attn_scores_masked_gen, dim=-1)
             attn_output_gen = attention_weights_gen @ v_gen
             attn_output_gen = attn_output_gen.permute(0, 2, 1, 3).contiguous().view(B_gen, T_gen, C_gen)
             mha_result_gen = mha_output_linears[i](attn_output_gen)
             x_gen = x_input_block_gen + mha_result_gen # Residual 1
             # Pre-LN FFN... (same logic as training)
             x_input_ffn_gen = x_gen
             x_ln2_gen = layer_norms_2[i](x_input_ffn_gen)
             ffn_hidden_gen = ffn_linear_1[i](x_ln2_gen)
             ffn_activated_gen = F.relu(ffn_hidden_gen)
             ffn_output_gen = ffn_linear_2[i](ffn_activated_gen)
             x_gen = x_input_ffn_gen + ffn_output_gen # Residual 2
        # Final Layers
        final_norm_output_gen = final_layer_norm(x_gen)
        logits_gen = output_linear_layer(final_norm_output_gen)

        # --- 3. Get Logits for Last Time Step ---
        logits_last_token = logits_gen[:, -1, :] # Focus on the prediction for the *next* token

        # --- 4. Apply Softmax -> Probabilities ---
        probs = F.softmax(logits_last_token, dim=-1)

        # --- 5. Sample Next Token ---
        next_token = torch.multinomial(probs, num_samples=1) # Sample based on probabilities

        # --- 6. Append Sampled Token ---
        generated_sequence = torch.cat((generated_sequence, next_token), dim=1)

print("\n--- Generation Complete ---")

# Decode the generated IDs back to text
final_generated_ids = generated_sequence[0].tolist()
decoded_text = ''.join([int_to_char[id] for id in final_generated_ids])

print(f"\nFinal Generated Text (including seed):")
print(decoded_text)


--- Step 5: Text Generation ---
Initial seed sequence: 't' -> [[31]]
Generating 200 new tokens...

--- Generation Complete ---

Final Generated Text (including seed):
tions in
it, 'and what is the use of a book,' thought Alice 'without pictures or
conversation?'
So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy a


In [29]:
# --- Load Saved Text Model State ---
print("\nStep 0.2: Loading pre-trained text model state...")
model_load_path = 'saved_models/transformer_model.pt'
if not os.path.exists(model_load_path):
    raise FileNotFoundError(f"Error: Model file not found at {model_load_path}. Please ensure 'transformer2.ipynb' was run and saved the model.")

loaded_state_dict = torch.load(model_load_path, map_location=device)
print(f"Loaded state dictionary from '{model_load_path}'.")

# --- Extract Config and Tokenizer ---
config = loaded_state_dict['config']
loaded_vocab_size = config['vocab_size']
d_model = config['d_model']
n_heads = config['n_heads']
n_layers = config['n_layers']
d_ff = config['d_ff']
loaded_block_size = config['block_size'] # Max sequence length for text model
d_k = d_model // n_heads

char_to_int = loaded_state_dict['tokenizer']['char_to_int']
int_to_char = loaded_state_dict['tokenizer']['int_to_char']

print("Extracted model configuration and tokenizer:")
print(f"  Loaded vocab_size: {loaded_vocab_size}")
print(f"  d_model: {d_model}")
# ... (print other loaded hyperparameters) ...
print(f"  Loaded block_size: {loaded_block_size}")


Step 0.2: Loading pre-trained text model state...


FileNotFoundError: Error: Model file not found at saved_models/transformer_model.pt. Please ensure 'transformer2.ipynb' was run and saved the model.

In [ ]:
print("\nStep 0.3: Defining special tokens and updating vocabulary...")

# --- Define Special Tokens ---
img_token = "<IMG>"
pad_token = "<PAD>"
eos_token = "<EOS>" # End-of-Sentence/Sequence
special_tokens = [img_token, pad_token, eos_token]

# --- Add Special Tokens to Vocabulary ---
current_vocab_size = loaded_vocab_size
for token in special_tokens:
    if token not in char_to_int:
        char_to_int[token] = current_vocab_size
        int_to_char[current_vocab_size] = token
        current_vocab_size += 1

# Update vocab_size
vocab_size = current_vocab_size
pad_token_id = char_to_int[pad_token] # Store the ID for later use

print(f"Added special tokens: {special_tokens}")
print(f"Updated vocabulary size: {vocab_size}")
print(f"PAD token ID: {pad_token_id}")

In [ ]:
print("\nStep 0.4: Defining sample multi-modal data...")

# --- Create Dummy Image Files ---
sample_data_dir = "sample_multimodal_data"
os.makedirs(sample_data_dir, exist_ok=True)
image_paths = {
    "red": os.path.join(sample_data_dir, "red_square.png"),
    "blue": os.path.join(sample_data_dir, "blue_square.png"),
    "green": os.path.join(sample_data_dir, "green_circle.png")
}
# (Code to create red, blue, green images - same as notebook)
img_red = Image.new('RGB', (64, 64), color = 'red')
img_red.save(image_paths["red"])
img_blue = Image.new('RGB', (64, 64), color = 'blue')
img_blue.save(image_paths["blue"])
img_green = Image.new('RGB', (64, 64), color = 'white')
from PIL import ImageDraw
draw = ImageDraw.Draw(img_green)
draw.ellipse((4, 4, 60, 60), fill='green', outline='green')
img_green.save(image_paths["green"])
print(f"Created dummy images in '{sample_data_dir}'.")

# --- Define Data Triplets ---
# Added <EOS> token to the end of responses.
sample_training_data = [
    {"image_path": image_paths["red"], "prompt": "What color is the shape?", "response": "red." + eos_token},
    # ... (other samples) ...
    {"image_path": image_paths["green"], "prompt": "Describe this.", "response": "a circle, it is green." + eos_token}
]
num_samples = len(sample_training_data)
print(f"Defined {num_samples} sample multi-modal data points.")

In [ ]:
print("\nStep 0.5: Loading pre-trained vision model (ResNet-18)...")

# --- Load Pre-trained ResNet-18 ---
vision_model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)

# --- Remove Final Classification Layer ---
vision_feature_dim = vision_model.fc.in_features # Get feature dimension (512)
vision_model.fc = nn.Identity() # Replace the classifier

# --- Set to Evaluation Mode and Move to Device ---
vision_model = vision_model.to(device)
vision_model.eval() # IMPORTANT: Disable dropout/batchnorm updates

print(f"Loaded ResNet-18 feature extractor.")
print(f"  Output feature dimension: {vision_feature_dim}")
print(f"  Vision model set to evaluation mode on device: {device}")


In [ ]:
print("\nStep 0.6: Defining image transformations...")

# --- Define Standard ImageNet Transforms ---
image_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # ImageNet mean/std
                         std=[0.229, 0.224, 0.225])
])

print("Defined image preprocessing pipeline (Resize, Crop, ToTensor, Normalize).")


In [ ]:
print("\nStep 0.7: Defining new/updated hyperparameters...")

# --- Multi-Modal Sequence Length ---
block_size = 64 # Increase block size for combined sequence
print(f"  Set combined block_size: {block_size}")

# --- Number of Image Tokens ---
num_img_tokens = 1
print(f"  Using {num_img_tokens} <IMG> token(s) to represent image features.")

# --- Training Parameters ---
learning_rate = 3e-4
batch_size = 4 # Reduce batch size
epochs = 2000
eval_interval = 500
print(f"  Updated Training Params: LR={learning_rate}, BatchSize={batch_size}, Epochs={epochs}")

# (Check if block_size is sufficient - code omitted for brevity)
min_req_block_size = num_img_tokens + max(len(d["prompt"]) + len(d["response"]) for d in sample_training_data) + 1
print(f"  Max sequence length in sample data (approx): {min_req_block_size}")
# Recreate the causal mask for the new block_size
causal_mask = torch.tril(torch.ones(block_size, block_size, device=device)).view(1, 1, block_size, block_size)
print(f"  Recreated causal mask for new block_size={block_size}")

In [ ]:
print("\nStep 1.1: Extracting image features for sample data...")
extracted_image_features = {}
unique_image_paths = set(d["image_path"] for d in sample_training_data)
print(f"Found {len(unique_image_paths)} unique images to process.")

for img_path in unique_image_paths:
    # --- Load Image ---
    img = Image.open(img_path).convert('RGB')
    # --- Apply Transformations ---
    img_tensor = image_transforms(img).unsqueeze(0).to(device)
    # --- Extract Features ---
    with torch.no_grad():
        feature_vector = vision_model(img_tensor)
    # --- Store Features ---
    extracted_image_features[img_path] = feature_vector.squeeze(0)
    print(f"  Extracted features for '{os.path.basename(img_path)}', shape: {extracted_image_features[img_path].shape}")

print("Finished extracting image features for all unique sample images.")

In [ ]:
print("\nStep 1.2: Tokenizing prompts and responses...")

# (Code to check for and add any new characters from data to vocab - from notebook)
current_vocab_size = vocab_size
all_chars = set()
for sample in sample_training_data:
    all_chars.update(sample["prompt"])
    response_text = sample["response"].replace(eos_token, "")
    all_chars.update(response_text)
new_chars_added = 0
for char in all_chars:
     if char not in char_to_int:
          # (Add char to mappings - code omitted)
          new_chars_added += 1
vocab_size = current_vocab_size + new_chars_added
print(f"Added {new_chars_added} new characters to vocabulary. New vocab_size: {vocab_size}")

In [ ]:
print("\nStep 1.3: Creating padded input/target sequences and masks...")

prepared_sequences = []
ignore_index = -100 # For loss calculation

for sample in tokenized_samples:
    # --- Construct Input Sequence IDs ---
    img_ids = [char_to_int[img_token]] * num_img_tokens
    # Input: <IMG> + prompt + response (except last token)
    input_ids_no_pad = img_ids + sample["prompt_ids"] + sample["response_ids"][:-1]

    # --- Construct Target Sequence IDs ---
    # Target: shifted input, ignore loss for <IMG> and prompt
    target_ids_no_pad = ([ignore_index] * len(img_ids)) + \
                         ([ignore_index] * len(sample["prompt_ids"])) + \
                         sample["response_ids"]

    # --- Padding ---
    current_len = len(input_ids_no_pad)
    pad_len = block_size - current_len
    # (Handle truncation if current_len > block_size - code omitted)
    input_ids = input_ids_no_pad + ([pad_token_id] * pad_len)
    target_ids = target_ids_no_pad + ([ignore_index] * pad_len) # Pad targets too

    # --- Create Attention Mask (Padding Mask) ---
    attention_mask = ([1] * current_len) + ([0] * pad_len) # 1 for real, 0 for pad

    # --- Store ---
    prepared_sequences.append({
        "image_path": sample["image_path"],
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "target_ids": torch.tensor(target_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long)
    })

# --- Stack into Tensors ---
all_input_ids = torch.stack([s['input_ids'] for s in prepared_sequences])
all_target_ids = torch.stack([s['target_ids'] for s in prepared_sequences])
all_attention_masks = torch.stack([s['attention_mask'] for s in prepared_sequences])
all_image_paths = [s['image_path'] for s in prepared_sequences] # Keep track of images

num_sequences_available = all_input_ids.shape[0]
print(f"Created {num_sequences_available} padded sequences with targets and masks.")
print(f"  Input IDs shape: {all_input_ids.shape}")
print(f"  Target IDs shape: {all_target_ids.shape}")
print(f"  Attention Mask shape: {all_attention_masks.shape}")


In [ ]:
print("\nStep 2.1: Re-initializing Embedding and Output Layers for new vocab size...")

# --- Token Embedding Table ---
new_token_embedding_table = nn.Embedding(vocab_size, d_model).to(device)
original_weights = loaded_state_dict['token_embedding_table']['weight'][:loaded_vocab_size, :]
with torch.no_grad():
    new_token_embedding_table.weight[:loaded_vocab_size, :] = original_weights
token_embedding_table = new_token_embedding_table
print(f"  Re-initialized Token Embedding Table, shape: {token_embedding_table.weight.shape}")

# --- Output Linear Layer ---
new_output_linear_layer = nn.Linear(d_model, vocab_size).to(device)
original_out_weight = loaded_state_dict['output_linear_layer']['weight'][:loaded_vocab_size, :]
original_out_bias = loaded_state_dict['output_linear_layer']['bias'][:loaded_vocab_size]
with torch.no_grad():
    new_output_linear_layer.weight[:loaded_vocab_size, :] = original_out_weight
    new_output_linear_layer.bias[:loaded_vocab_size] = original_out_bias
output_linear_layer = new_output_linear_layer
print(f"  Re-initialized Output Linear Layer, weight shape: {output_linear_layer.weight.shape}")


In [ ]:
print("\nStep 2.3: Loading parameters for existing Transformer Blocks...")

# (Code to re-initialize layer lists and load state_dict for each component:
# layer_norms_1, mha_qkv_linears, mha_output_linears, layer_norms_2,
# ffn_linear_1, ffn_linear_2, final_layer_norm - from notebook)

# Reload components and load state dictionaries from loaded_state_dict
layer_norms_1 = []
mha_qkv_linears = []
mha_output_linears = []
layer_norms_2 = []
ffn_linear_1 = []
ffn_linear_2 = []

for i in range(n_layers):
    ln1 = nn.LayerNorm(d_model).to(device)
    ln1.load_state_dict(loaded_state_dict['layer_norms_1'][i])
    layer_norms_1.append(ln1)

    qkv_dict = loaded_state_dict['mha_qkv_linears'][i]
    has_qkv_bias = 'bias' in qkv_dict
    qkv = nn.Linear(d_model, 3 * d_model, bias=has_qkv_bias).to(device)
    qkv.load_state_dict(qkv_dict)
    mha_qkv_linears.append(qkv)

    out_dict = loaded_state_dict['mha_output_linears'][i]
    has_out_bias = 'bias' in out_dict
    out = nn.Linear(d_model, d_model, bias=has_out_bias).to(device)
    out.load_state_dict(out_dict)
    mha_output_linears.append(out) # Corrected variable name

    ln2 = nn.LayerNorm(d_model).to(device)
    ln2.load_state_dict(loaded_state_dict['layer_norms_2'][i])
    layer_norms_2.append(ln2)

    ff1_dict = loaded_state_dict['ffn_linear_1'][i]
    has_ff1_bias = 'bias' in ff1_dict
    ff1 = nn.Linear(d_model, d_ff, bias=has_ff1_bias).to(device)
    ff1.load_state_dict(ff1_dict)
    ffn_linear_1.append(ff1)

    ff2_dict = loaded_state_dict['ffn_linear_2'][i]
    has_ff2_bias = 'bias' in ff2_dict
    ff2 = nn.Linear(d_ff, d_model, bias=has_ff2_bias).to(device)
    ff2.load_state_dict(ff2_dict)
    ffn_linear_2.append(ff2)
    print(f"  Loaded components for Layer {i+1}/{n_layers}.")

final_layer_norm = nn.LayerNorm(d_model).to(device)
final_layer_norm.load_state_dict(loaded_state_dict['final_layer_norm'])
print("  Loaded Final LayerNorm.")

# Load Positional Encoding
positional_encoding = loaded_state_dict['positional_encoding'].to(device)
# (Code to recompute PE if block_size changed - from notebook)
if positional_encoding.shape[1] != block_size:
     print(f"Warning: Loaded positional encoding size ({positional_encoding.shape[1]}) != new block_size ({block_size}). Recomputing.")
     # (Recompute PE logic - code omitted)
     new_pe = torch.zeros(block_size, d_model, device=device)
     position = torch.arange(0, block_size, dtype=torch.float, device=device).unsqueeze(1)
     div_term_indices = torch.arange(0, d_model, 2, dtype=torch.float, device=device)
     div_term = torch.exp(div_term_indices * (-math.log(10000.0) / d_model))
     new_pe[:, 0::2] = torch.sin(position * div_term)
     new_pe[:, 1::2] = torch.cos(position * div_term)
     positional_encoding = new_pe.unsqueeze(0)
     print(f"  Recomputed Positional Encoding matrix, shape: {positional_encoding.shape}")


print("Finished loading existing model components.")

In [ ]:
print("\nStep 2.3: Loading parameters for existing Transformer Blocks...")

# (Code to re-initialize layer lists and load state_dict for each component:
# layer_norms_1, mha_qkv_linears, mha_output_linears, layer_norms_2,
# ffn_linear_1, ffn_linear_2, final_layer_norm - from notebook)

# Reload components and load state dictionaries from loaded_state_dict
layer_norms_1 = []
mha_qkv_linears = []
mha_output_linears = []
layer_norms_2 = []
ffn_linear_1 = []
ffn_linear_2 = []

for i in range(n_layers):
    ln1 = nn.LayerNorm(d_model).to(device)
    ln1.load_state_dict(loaded_state_dict['layer_norms_1'][i])
    layer_norms_1.append(ln1)

    qkv_dict = loaded_state_dict['mha_qkv_linears'][i]
    has_qkv_bias = 'bias' in qkv_dict
    qkv = nn.Linear(d_model, 3 * d_model, bias=has_qkv_bias).to(device)
    qkv.load_state_dict(qkv_dict)
    mha_qkv_linears.append(qkv)

    out_dict = loaded_state_dict['mha_output_linears'][i]
    has_out_bias = 'bias' in out_dict
    out = nn.Linear(d_model, d_model, bias=has_out_bias).to(device)
    out.load_state_dict(out_dict)
    mha_output_linears.append(out) # Corrected variable name

    ln2 = nn.LayerNorm(d_model).to(device)
    ln2.load_state_dict(loaded_state_dict['layer_norms_2'][i])
    layer_norms_2.append(ln2)

    ff1_dict = loaded_state_dict['ffn_linear_1'][i]
    has_ff1_bias = 'bias' in ff1_dict
    ff1 = nn.Linear(d_model, d_ff, bias=has_ff1_bias).to(device)
    ff1.load_state_dict(ff1_dict)
    ffn_linear_1.append(ff1)

    ff2_dict = loaded_state_dict['ffn_linear_2'][i]
    has_ff2_bias = 'bias' in ff2_dict
    ff2 = nn.Linear(d_ff, d_model, bias=has_ff2_bias).to(device)
    ff2.load_state_dict(ff2_dict)
    ffn_linear_2.append(ff2)
    print(f"  Loaded components for Layer {i+1}/{n_layers}.")

final_layer_norm = nn.LayerNorm(d_model).to(device)
final_layer_norm.load_state_dict(loaded_state_dict['final_layer_norm'])
print("  Loaded Final LayerNorm.")

# Load Positional Encoding
positional_encoding = loaded_state_dict['positional_encoding'].to(device)
# (Code to recompute PE if block_size changed - from notebook)
if positional_encoding.shape[1] != block_size:
     print(f"Warning: Loaded positional encoding size ({positional_encoding.shape[1]}) != new block_size ({block_size}). Recomputing.")
     # (Recompute PE logic - code omitted)
     new_pe = torch.zeros(block_size, d_model, device=device)
     position = torch.arange(0, block_size, dtype=torch.float, device=device).unsqueeze(1)
     div_term_indices = torch.arange(0, d_model, 2, dtype=torch.float, device=device)
     div_term = torch.exp(div_term_indices * (-math.log(10000.0) / d_model))
     new_pe[:, 0::2] = torch.sin(position * div_term)
     new_pe[:, 1::2] = torch.cos(position * div_term)
     positional_encoding = new_pe.unsqueeze(0)
     print(f"  Recomputed Positional Encoding matrix, shape: {positional_encoding.shape}")


print("Finished loading existing model components.")

In [ ]:
# Set layers to train mode
token_embedding_table.train()
vision_projection_layer.train()
for layer in layer_norms_1:
    layer.train()
final_layer_norm.train()
output_linear_layer.train()

for epoch in range(epochs):
    # Batch Selection
    indices = torch.randint(0, num_sequences_available, (batch_size,))
    xb_ids, yb_ids, batch_masks = all_input_ids[indices].to(device), all_target_ids[indices].to(device), all_attention_masks[indices].to(device)
    batch_img_paths = [all_image_paths[i] for i in indices.tolist()]
    try:
        batch_img_features = torch.stack([extracted_image_features[p] for p in batch_img_paths]).to(device)
    except KeyError:
        continue

    # Forward Pass
    projected_img_features = vision_projection_layer(batch_img_features).unsqueeze(1)
    text_token_embeddings = token_embedding_table(xb_ids)
    combined_embeddings = text_token_embeddings.clone()
    combined_embeddings[:, :num_img_tokens, :] = projected_img_features
    x = combined_embeddings + positional_encoding[:, :xb_ids.shape[1], :]

    # Attention Mask
    combined_attn_mask = causal_mask[:, :xb_ids.shape[1], :xb_ids.shape[1]] * batch_masks.unsqueeze(1).unsqueeze(2)

    # Transformer Blocks
    for i in range(n_layers):
        x_ln1 = layer_norms_1[i](x)
        qkv = mha_qkv_linears[i](x_ln1).view(B, T, n_heads, 3 * d_k).permute(0, 2, 1, 3)
        q, k, v = qkv.chunk(3, dim=-1)
        attn_scores = (q @ k.transpose(-2, -1)) * (d_k ** -0.5)
        attn_scores_masked = attn_scores.masked_fill(combined_attn_mask == 0, float('-inf'))
        attention_weights = torch.nan_to_num(F.softmax(attn_scores_masked, dim=-1))
        attn_output = (attention_weights @ v).permute(0, 2, 1, 3).contiguous().view(B, T, C)
        x = x + mha_output_linears[i](attn_output)

        # FFN
        x = x + ffn_linear_2[i](F.relu(ffn_linear_1[i](layer_norms_2[i](x))))

    # Final Layers
    logits = output_linear_layer(final_layer_norm(x))

    # Calculate Loss
    targets_reshaped = yb_ids.view(-1) if yb_ids.size(1) == logits.shape[1] else yb_ids[:, :logits.shape[1]].view(-1)
    loss = criterion(logits.view(-1, V_loss), targets_reshaped)

    # Backpropagation
    optimizer.zero_grad()
    if torch.isfinite(loss):
        loss.backward()
        optimizer.step()

    # Logging
    if loss is not None:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

print("--- Multi-Modal Training Loop Completed ---")

# Testing Chat with Images Functionality

In [ ]:
print("\nStep 4.3: Decoding generated sequence...")

if generated_sequence_ids is not None:
    final_ids_list = generated_sequence_ids[0].tolist()
    decoded_text = "".join([int_to_char.get(id_val, f"[UNK:{id_val}]") for id_val in final_ids_list]) # Use .get for safety

    print(f"\n--- Final Generated Output ---")
    print(f"Image: {os.path.basename(test_image_path)}")
    response_start_index = num_img_tokens + len(test_prompt_text)
    print(f"Prompt: {test_prompt_text}")
    print(f"Generated Response: {decoded_text[response_start_index:]}")
else:
    print("Decoding skipped.")

In [ ]:
print("\n--- Conceptual Video Handling ---")

# 0. Add <VID> token
vid_token = "<VID>"
if vid_token not in char_to_int:
    char_to_int[vid_token] = vocab_size
    int_to_char[vocab_size] = vid_token
    vocab_size += 1
    print(f"Added {vid_token}. New vocab_size: {vocab_size}")
    # Need to resize embedding/output layers again if doing this properly!

# 1. Load video frames (dummy example)
video_path = "path/to/dummy_video.mp4"
# dummy_frames = load_video_frames(video_path, num_frames=16) # List of PIL Images
# Create dummy frames for illustration: list of 16 green PIL Images
dummy_frames = [img_green] * 16 # Reuse green circle image
print(f"Loaded {len(dummy_frames)} dummy frames for '{video_path}'.")

# 2. Extract features for each frame
frame_features_list = []
with torch.no_grad():
    for frame_img in dummy_frames:
        # Apply same image transforms as before
        frame_tensor = image_transforms(frame_img).unsqueeze(0).to(device)
        # Use the SAME vision model
        frame_feature = vision_model(frame_tensor) # (1, vision_feature_dim)
        frame_features_list.append(frame_feature)

# Stack features: (num_frames, vision_feature_dim)
all_frame_features = torch.cat(frame_features_list, dim=0)
print(f"Extracted frame features, shape: {all_frame_features.shape}") # e.g., (16, 512)

# 3. Combine Features (Simple Averaging)
video_feature_avg = torch.mean(all_frame_features, dim=0, keepdim=True) # (1, vision_feature_dim)
print(f"Averaged video features, shape: {video_feature_avg.shape}") # e.g., (1, 512)

# 4. Project Video Features
# Option 1: Use the same projection layer as images (if appropriate)
# Option 2: Create a dedicated video projection layer
# vision_video_projection_layer = nn.Linear(vision_feature_dim, d_model).to(device)
# Initialize and train this layer appropriately!
# For simplicity here, let's reuse the image projection layer conceptually
with torch.no_grad(): # Assuming projection layer is trained/loaded
    projected_video_feature = vision_projection_layer(video_feature_avg) # (1, d_model)
print(f"Projected video feature, shape: {projected_video_feature.shape}") # e.g., (1, 64)

# 5. Prepare Input Sequence (Example)
prompt_text = "What happens in the video?"
prompt_ids = [char_to_int[ch] for ch in prompt_text]
vid_id = char_to_int[vid_token]

# Input: [<VID>, prompt tokens]
input_ids_vid = torch.tensor([[vid_id] + prompt_ids], dtype=torch.long, device=device)
print(f"Example input sequence with video: {input_ids_vid.tolist()}")

# During the actual forward pass for this sequence, the embedding for vid_id
# would be replaced by projected_video_feature.

In [ ]:
# Assume dummy_audio_lib exists
# from dummy_audio_lib import load_audio_waveform, compute_mfccs

print("\n--- Conceptual Audio Handling ---")

# 0. Add <AUD> token
aud_token = "<AUD>"
if aud_token not in char_to_int:
    char_to_int[aud_token] = vocab_size
    int_to_char[vocab_size] = aud_token
    vocab_size += 1
    print(f"Added {aud_token}. New vocab_size: {vocab_size}")
    # Resize embedding/output layers needed!

# 1. Load Audio (dummy waveform)
audio_path = "path/to/dummy_audio.wav"
# waveform, sample_rate = load_audio_waveform(audio_path)
# Dummy: 1 second of audio at 16kHz
sample_rate = 16000
waveform = np.random.randn(1 * sample_rate)
print(f"Loaded dummy waveform for '{audio_path}', length: {len(waveform)}")

# 2. Extract Features (dummy MFCCs + Linear layer)
# mfccs = compute_mfccs(waveform, sample_rate) # Shape: (num_frames, num_mfcc_coeffs)
# Dummy: (99 time steps, 13 coefficients)
mfccs = np.random.randn(99, 13)
mfccs_tensor = torch.tensor(mfccs, dtype=torch.float).to(device)
print(f"Computed dummy MFCC features, shape: {mfccs_tensor.shape}")

# Simple placeholder feature extractor (e.g., a Linear layer over coefficients)
# Needs to be defined and potentially trained
audio_feature_dim = mfccs_tensor.shape[1] # e.g., 13
dummy_audio_extractor = nn.Linear(audio_feature_dim, 256).to(device) # Example: map MFCCs -> 256
# Initialize/load weights for dummy_audio_extractor!
with torch.no_grad():
     audio_features_time = dummy_audio_extractor(mfccs_tensor) # (num_frames, 256)
print(f"Extracted dummy audio features over time, shape: {audio_features_time.shape}")

# 3. Combine Features (Simple Averaging over time)
audio_feature_avg = torch.mean(audio_features_time, dim=0, keepdim=True) # (1, 256)
print(f"Averaged audio features, shape: {audio_feature_avg.shape}")

# 4. Project Audio Features to d_model
# Need a dedicated audio projection layer
# audio_projection_layer = nn.Linear(256, d_model).to(device)
# Initialize and train this layer!
# For simplicity, reuse image projection layer conceptually (dimensions likely won't match in reality!)
# Assuming audio_feature_avg was projected to vision_feature_dim somehow or using a dedicated layer:
# Dummy projection to d_model:
dummy_audio_projection = nn.Linear(256, d_model).to(device)
with torch.no_grad():
    projected_audio_feature = dummy_audio_projection(audio_feature_avg) # (1, d_model)
print(f"Projected audio feature, shape: {projected_audio_feature.shape}") # e.g., (1, 64)

# 5. Prepare Input Sequence (Example)
prompt_text = "What sound is this?"
prompt_ids = [char_to_int[ch] for ch in prompt_text]
aud_id = char_to_int[aud_token]

# Input: [<AUD>, prompt tokens]
input_ids_aud = torch.tensor([[aud_id] + prompt_ids], dtype=torch.long, device=device)
print(f"Example input sequence with audio: {input_ids_aud.tolist()}")

# During the forward pass, the embedding for aud_id would be replaced by projected_audio_feature.


In [ ]:
# --- Load Saved Multi-Modal Model State ---
print("\nStep 0.2: Loading state from multi-modal model...")
model_load_path = 'saved_models/multimodal_model.pt'
# (Check if file exists - code omitted)
if not os.path.exists(model_load_path):
    raise FileNotFoundError(f"Error: Model file not found at {model_load_path}. Please ensure 'multimodal.ipynb' was run and saved the model.")

loaded_state_dict = torch.load(model_load_path, map_location=device)
print(f"Loaded state dictionary from '{model_load_path}'.")

# --- Extract Config and Tokenizer ---
config = loaded_state_dict['config']
vocab_size = config['vocab_size'] # Includes special tokens now
d_model = config['d_model']
n_layers = config['n_layers']
n_heads = config['n_heads']
d_ff = config['d_ff']
block_size = config['block_size'] # Use the size from multimodal setup
vision_feature_dim = config['vision_feature_dim'] # e.g., 512
d_k = d_model // n_heads

char_to_int = loaded_state_dict['tokenizer']['char_to_int']
int_to_char = loaded_state_dict['tokenizer']['int_to_char']
pad_token_id = char_to_int.get('<PAD>', -1)
print("Extracted model configuration and tokenizer:")
print(f"  vocab_size: {vocab_size}")
# (Print other config values...)
print(f"  block_size: {block_size}")
print(f"  vision_feature_dim: {vision_feature_dim}")
print(f"  PAD token ID: {pad_token_id}")


# --- Load Positional Encoding ---
positional_encoding = loaded_state_dict['positional_encoding'].to(device)
print(f"Loaded positional encoding with shape: {positional_encoding.shape}")

# --- Store State Dicts for Text Components ---
# (Store state dicts for embedding, layer_norms, attention, ffn, final_norm - code omitted)
loaded_embedding_dict = loaded_state_dict['token_embedding_table']
loaded_ln1_dicts = loaded_state_dict['layer_norms_1']
loaded_qkv_dicts = loaded_state_dict['mha_qkv_linears']
loaded_mha_out_dicts = loaded_state_dict['mha_output_linears']
loaded_ln2_dicts = loaded_state_dict['layer_norms_2']
loaded_ffn1_dicts = loaded_state_dict['ffn_linear_1']
loaded_ffn2_dicts = loaded_state_dict['ffn_linear_2']
loaded_final_ln_dict = loaded_state_dict['final_layer_norm']
print("Stored state dicts for text transformer components.")

# --- Load Vision Feature Extractor (Frozen) ---
print("Loading pre-trained vision model (ResNet-18) for target feature extraction...")
vision_model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
vision_model.fc = nn.Identity() # Remove classifier
vision_model = vision_model.to(device)
vision_model.eval() # Keep frozen
for param in vision_model.parameters():
    param.requires_grad = False
print(f"Loaded and froze ResNet-18 feature extractor on device: {device}")

# --- Define Image Transformations (same as before) ---
# (Define image_transforms - code omitted)
image_transforms = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print("Defined image transformations.")

In [ ]:
print("\nStep 0.3: Defining sample text-to-image data...")

# --- Image Paths (Assuming they exist) ---
# (Define image_paths dictionary and check/recreate files - code omitted)
sample_data_dir = "sample_multimodal_data"
image_paths = {
    "red_square": os.path.join(sample_data_dir, "red_square.png"),
    "blue_square": os.path.join(sample_data_dir, "blue_square.png"),
    "green_circle": os.path.join(sample_data_dir, "green_circle.png")
}
# (Code to check/recreate images)


# --- Define Text Prompt -> Image Path Pairs ---
text_to_image_data = [
    {"prompt": "a red square", "image_path": image_paths["red_square"]},
    {"prompt": "the square is red", "image_path": image_paths["red_square"]},
    {"prompt": "show a blue square", "image_path": image_paths["blue_square"]},
    {"prompt": "blue shape, square", "image_path": image_paths["blue_square"]},
    {"prompt": "a green circle", "image_path": image_paths["green_circle"]},
    {"prompt": "the circle, it is green", "image_path": image_paths["green_circle"]},
    {"prompt": "make a square that is red", "image_path": image_paths["red_square"]}
]
num_samples = len(text_to_image_data)

In [ ]:
print("\nStep 0.4: Extracting target image features...")
target_image_features = {} # Dict: {image_path: feature_tensor}
known_features_list = [] # List: [(path, feature_tensor)]

# --- Loop Through Unique Image Paths ---
unique_image_paths_in_data = sorted(list(set(d["image_path"] for d in text_to_image_data)))
print(f"Found {len(unique_image_paths_in_data)} unique target images to process.")

for img_path in unique_image_paths_in_data:
    # --- Load Image ---
    img = Image.open(img_path).convert('RGB')
    # --- Apply Transformations ---
    img_tensor = image_transforms(img).unsqueeze(0).to(device)
    # --- Extract Features (using frozen vision_model) ---
    with torch.no_grad():
        feature_vector = vision_model(img_tensor)
    feature_vector_squeezed = feature_vector.squeeze(0)
    print(f"  Extracted features for '{os.path.basename(img_path)}', shape: {feature_vector_squeezed.shape}")

    # --- Store Features ---
    target_image_features[img_path] = feature_vector_squeezed
    known_features_list.append((img_path, feature_vector_squeezed))

# (Error check if target_image_features is empty - code omitted)
if not target_image_features:
     raise ValueError("No target image features were extracted. Cannot proceed.")

print("Finished extracting and storing target image features.")
print(f"Stored {len(known_features_list)} known (path, feature) pairs for generation lookup.")

In [ ]:
print("\nStep 0.5: Defining training hyperparameters for text-to-image...")
# (Check block_size vs max_prompt_len - code omitted)
print(f"Using block_size: {block_size}")
# Recreate causal mask for this block size
causal_mask = torch.tril(torch.ones(block_size, block_size, device=device)).view(1, 1, block_size, block_size)

learning_rate = 1e-4 # Lower LR for fine-tuning
batch_size = 4
epochs = 5000
eval_interval = 500
print(f"  Training Params: LR={learning_rate}, BatchSize={batch_size}, Epochs={epochs}")